# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibly one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probability of net winning or losing after 50 rounds?


9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 

In [ ]:
import random
class Deck():
    suits = ["♣️", "♦️", "❤️", "♠️"]
    values = list(range(2,11)) + ["Jack", "Queen", "King", "Ace"]
    def __init__(self, num_decks: int = 6):
        self.num_decks = num_decks
        self.cards = []
        self.plastic_card = False  
        self.deck = self.make_deck()
        self.reset()
    def make_deck(self):
        deck = list()
        for value in self.values:
            for suit in self.suits:
                deck.append(str(value) + " of " + str(suit))
        return deck * self.num_decks
    def get_values(val):
        if val in ["Jack", "Queen", "King"]:
            return 10
        elif val == "Ace":
            return 11
        else:
            return int(val)
    def reset(self):
        self.deck = self.make_deck()
        self.shuffle()
        min_index = int(len(self.deck) * 0.30)
        max_index = int(len(self.deck) * 0.70)
        self.plastic_card_location = random.randint(min_index, max_index)
        self.plastic_card = False
    def shuffle(self):
        random.shuffle(self.deck)
        
    def draw_card(self):
        if not self.deck:
            self.reset()
        if len(self.deck) <= (len(self.deck) - self.plastic_card_location):
            self.plastic_card = True
            print("Shuffeling deck")
        return self.deck.pop(0)
class Hand:
    def __init__(self):
        self.deck = []
    def hit(self, card):
        self.deck.append(card)
    def get_value(self):
        value = 0
        aces = 0
        for card in self.deck:
            values = str(card).split(" ")[0]
            if values in ["Jack", "Queen", "King"]:
                value += 10
            elif values == "Ace":
                aces += 1
                value += 11
            else:
                value += int(values)
        while value > 21 and aces > 0:
            value -= 10
            aces -= 1
        return value
        pass
    def bust(self):
        return self.get_value() > 21
    def __str__(self):
        cards_str = ", ".join(str(card) for card in self.deck)
        return f"\n{cards_str}\n---Total: {self.get_value()}---\n"
class Player:
    def __init__(self, chips = 500):
        self.chips = chips
        self.hand = Hand()
        self.bet = 0
    def place_bet(self, amount):
        if amount <= self.chips:
            self.bet = amount
            self.chips -= amount
        else:
            self.bet = self.chips
            self.chips = 0
    def bust(self):
        return self.hand.get_value() > 21
    def win(self):
        print("\n---The Player wins the round---\n")
        self.chips += self.bet * 2
        self.bet = 0
    def push(self):
        print("\n---Push---\n")
        self.chips += self.bet
        self.bet = 0
    def loss(self):
        print("\n---The Player losses the round---\n")
        self.bet = 0
    def choose_action(self, dealer_up_card: Deck) -> str:
        while True:
            choice = input("\n---What do you want to do? |Hit| |Stand|---\n").strip().upper()
            if choice in ['H', 'S']:
                return choice
            print("XXX Invalid choice XXX \nUse H or S")
class Com:
    def __init__(self,chips = 500):
        self.chips = chips
        self.hand = Hand()
        self.bet = 0
    def place_bet(self, amount: int):
        if amount <= self.chips:
            self.current_bet = amount
            self.chips -= amount
        else:
            self.current_bet = self.chips
            self.chips = 0
    def win(self):
        self.chips += self.bet * 2
        self.bet = 0
    def push(self):
        self.chips += self.bet
        self.bet = 0
    def loss(self):
        self.bet = 0
    def choose_action(self, dealer_up_card = None):
        raise NotImplementedError
    
class Dealer(Com):
    def __init__(self):
        super().__init__(chips = 0)
    def choose_action(self, dealer_up_card = None):
        if self.hand.get_value() < 17:
            return 'H'
        return 'S'
class Blackjack:
    def __init__(self, chips: int = 1000):
        self.deck = Deck()
        self.player = Player(chips)
        self.dealer = Dealer()
    def play(self):
        print("\n---Table is open. Place your bets.---\n")
        if self.deck.plastic_card:
            print("---Deck has been shuffled---")
            self.deck.reset()
        bet = int(input("Enter the amount you want to bet: "))
        self.player.place_bet(bet)
        self.player.hand = Hand()
        self.dealer.hand = Hand()
        self.player.hand.hit(self.deck.draw_card())
        self.dealer.hand.hit(self.deck.draw_card())
        self.player.hand.hit(self.deck.draw_card())
        self.dealer.hand.hit(self.deck.draw_card())
        dealer_up_card = self.dealer.hand.deck[0]
        print(f"\nDealer's cards: {dealer_up_card}")
        print(f"\nYour cards: {self.player.hand}")
        while not self.player.hand.bust():
            action = self.player.choose_action(dealer_up_card)
            if action == 'H':
                self.player.hand.hit(self.deck.draw_card())
                print("\nYou have: ",self.player.hand)
            else:
                break
        if not self.player.hand.bust():
            print("\nDealer reveals card: ", self.dealer.hand.deck)
            while not self.dealer.hand.bust():
                action = self.dealer.choose_action()
                if action == 'H':
                    print("\n---Dealer hits---")
                    self.dealer.hand.hit(self.deck.draw_card())
                    print("\nDealer's hand: ", self.dealer.hand.deck)
                else:
                    print("\n---Dealer stands---")
                    break
        self.results()
    def results(self):
        player_score = self.player.hand.get_value()
        dealer_score = self.dealer.hand.get_value()
        if self.player.hand.bust():
            self.player.loss()
        elif self.dealer.hand.bust():
            self.player.win()
        elif player_score > dealer_score:
            print("\nPlayer has: ",{player_score},"\nDealer has: ",{dealer_score})
            self.player.win()
        elif player_score < dealer_score:
            print("\nPlayer has: ",{player_score},"\nDealer has: ",{dealer_score})
            self.player.loss()
        else:
            print("\nPlayer has: ",{player_score},"\nDealer has: ",{dealer_score})
            self.player.push()
    
#my_hand = Deck()
#print(Deck.get_values("Jack"))
#print(my_hand.deck[:12])
#drawn_card = my_hand.draw_card()
#print(drawn_card)
#print(len(my_hand.deck))
#print(my_hand.plastic_card)
if __name__ == "__main__":
    game = Blackjack(chips = 500)
    for _ in range(2):
        if game.player.chips <= 0:
            print("Out of chips")
            break
        game.play()


---Table is open. Place your bets.---



Enter the amount you want to bet:  200



Dealer's cards: 5 of ♠️

Your cards: 
Ace of ❤️, Queen of ♣️
---Total: 21---




---What do you want to do? |Hit| |Stand|---
 S



Dealer reveals card:  ['5 of ♠️', '4 of ♣️']

---Dealer hits---

Dealer's hand:  ['5 of ♠️', '4 of ♣️', '6 of ♠️']

---Dealer hits---

Dealer's hand:  ['5 of ♠️', '4 of ♣️', '6 of ♠️', 'King of ❤️']

---The Player wins the round---


---Table is open. Place your bets.---

